# Medical Insurance Cost Prediction
Full workflow: data loading, EDA, preprocessing, regression model comparison, evaluation and saving outputs.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
URL="https://raw.githubusercontent.com/JovianML/opendatasets/master/data/medical-charges.csv"
Path("data").mkdir(exist_ok=True)
raw=Path("data/medical_insurance_raw.csv")
df=pd.read_csv(raw if raw.exists() else URL)
if not raw.exists(): df.to_csv(raw,index=False)
print(df.shape); display(df.head()); df.info()

## Dataset checks and EDA

In [ ]:
display(df.describe(include="all").T)
display(df.isna().sum().to_frame("missing values"))
print("Duplicate rows:",df.duplicated().sum())
df.hist(figsize=(12,8),bins=25); plt.tight_layout(); plt.show()
sns.boxplot(data=df,x="smoker",y="charges"); plt.title("Charges by smoking status"); plt.show()
sns.heatmap(df.select_dtypes(include=np.number).corr(),annot=True,cmap="coolwarm"); plt.show()

## Preprocessing and train/test split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
import joblib, json
df.columns=df.columns.str.strip().str.lower().str.replace(r"[^a-z0-9]+","_",regex=True).str.strip("_")
for c in ["sex","smoker","region"]: df[c]=df[c].astype("string").str.strip().str.lower()
for c in ["age","bmi","children","charges"]: df[c]=pd.to_numeric(df[c],errors="coerce")
n0=len(df); dup=int(df.duplicated().sum()); df=df.drop_duplicates().dropna(subset=["charges"]).copy()
df.to_csv("data/medical_insurance_cleaned.csv",index=False)
X=df.drop(columns="charges"); y=df.charges
num=["age","bmi","children"]; cat=["sex","smoker","region"]
prep=ColumnTransformer([("num",Pipeline([("imp",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num),
("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("ohe",OneHotEncoder(handle_unknown="ignore"))]),cat)])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42)
print("Rows initially:",n0,"duplicates removed:",dup,"clean rows:",len(df))

## Train models and evaluate

In [ ]:
models={"Linear Regression":LinearRegression(),
"Decision Tree":DecisionTreeRegressor(max_depth=5,random_state=42),
"Random Forest":RandomForestRegressor(n_estimators=300,min_samples_leaf=2,random_state=42,n_jobs=-1),
"Gradient Boosting":GradientBoostingRegressor(random_state=42)}
trained={}; rows=[]
for name,est in models.items():
 pipe=Pipeline([("prep",prep),("model",est)]); pipe.fit(X_train,y_train); pred=pipe.predict(X_test)
 rows.append({"Model":name,"MAE":mean_absolute_error(y_test,pred),"MSE":mean_squared_error(y_test,pred),
 "RMSE":np.sqrt(mean_squared_error(y_test,pred)),"R2":r2_score(y_test,pred)})
 trained[name]=pipe
results=pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True); display(results)
Path("outputs").mkdir(exist_ok=True); results.to_csv("outputs/model_comparison.csv",index=False)
best_name=results.iloc[0]["Model"]; best=trained[best_name]
Path("models").mkdir(exist_ok=True); joblib.dump(best,"models/best_insurance_model.joblib")
print("Best model by test RMSE:",best_name)

## Actual vs predicted plot and next steps
Use the results from your run in your report. MAE/RMSE lower is better; R² closer to 1 generally indicates better fit. Consider cross-validation, tuning, residual analysis and feature importance as extensions.

In [ ]:
pred=best.predict(X_test)
plt.scatter(y_test,pred,alpha=.6); lo=min(y_test.min(),pred.min()); hi=max(y_test.max(),pred.max())
plt.plot([lo,hi],[lo,hi],"r--"); plt.xlabel("Actual charges"); plt.ylabel("Predicted charges"); plt.title(best_name); plt.show()
residual=y_test-pred
sns.histplot(residual,kde=True); plt.title("Residual distribution"); plt.show()